In [ ]:
## App

In [1]:
#Cell 1

import csv
import math
import multiprocessing as mp
import os
import re
import time
import unicodedata
from collections import Counter, defaultdict

import lightgbm as lgb
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler, Levenshtein
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold

SEED = 42
T0 = time.time()

def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

# Ensure output directories exist
os.makedirs("output", exist_ok=True)
os.makedirs("work_dir", exist_ok=True)

In [2]:
#Cell2

def read_tsv(path):
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, quoting=csv.QUOTE_NONE)
    with open(path, encoding="utf-8") as f:
        n_lines = sum(1 for line in f if line.strip()) - 1
    if n_lines != len(df):
        log(f"WARNING {path}: {n_lines} lines but {len(df)} rows parsed - inspect quoting/tabs")
    return df

def write_id_lists(path, s1_ids, mapping, col):
    with open(path, "w", encoding="utf-8", newline="") as f:
        f.write(f"source1_entity_id\t{col}\n")
        for s in s1_ids:
            ids = list(dict.fromkeys(mapping.get(s, [])))
            f.write(f"{s}\t{','.join(ids)}\n")

TABLE = {
    "incorporated": "inc", "corporation": "corp", "company": "co", "limited": "ltd", "private": "pvt",
    "compagnie": "cie", "societe": "ste", "etablissements": "ets", "etablissement": "ets",
    "enterprises": "ent", "enterprise": "ent", "industries": "ind", "industry": "ind",
    "international": "intl", "technologies": "tech", "technology": "tech", "services": "svc",
    "service": "svc", "brothers": "bros", "manufacturing": "mfg", "associates": "assoc",
    "solutions": "sol", "solution": "sol", "shree": "sri", "shri": "sri", "sree": "sri",
    "et": "and", "und": "and", "y": "and",
    "street": "st", "saint": "st", "sainte": "ste", "road": "rd", "avenue": "ave", "av": "ave",
    "boulevard": "blvd", "bd": "blvd", "drive": "dr", "lane": "ln", "place": "pl", "suite": "ste",
    "floor": "fl", "flr": "fl", "building": "bldg", "apartment": "apt", "appartement": "apt",
    "highway": "hwy", "parkway": "pkwy", "court": "ct", "square": "sq", "sector": "sec",
    "near": "nr", "opposite": "opp", "north": "n", "south": "s", "east": "e", "west": "w",
    "number": "no", "num": "no", "chemin": "ch", "route": "rte", "impasse": "imp", "allee": "all",
    "faubourg": "fbg", "centre": "ctr", "center": "ctr", "mount": "mt", "district": "dist",
}
LEGAL = {"inc", "corp", "co", "ltd", "pvt", "llc", "llp", "lp", "plc", "pllc", "pte", "opc", "sarl",
         "sas", "sasu", "sa", "eurl", "sci", "snc", "scop", "selarl", "gmbh", "ag", "bv", "srl", "spa",
         "cie", "ste", "ets"}
NAME_STOP = {"and", "the", "of", "de", "du", "des", "la", "le", "les", "l", "d", "s"}
LANDMARK = {"nr", "opp", "behind", "beside", "next", "adjacent", "facing", "pres", "cote", "vis"}

_re_ms = re.compile(r"\bm\s*/\s*s\b")
_re_pin = re.compile(r"\b(\d{3})[\s-](\d{3})\b")
_re_nonalnum = re.compile(r"[^0-9a-z]+")

def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def tokens(s):
    s = strip_accents(str(s)).lower()
    s = _re_ms.sub(" ", s).replace("&", " and ").replace("'", " ").replace("\u2019", " ")
    s = _re_pin.sub(r"\1\2", s)
    return [TABLE.get(t, t) for t in _re_nonalnum.sub(" ", s).split()]

def merge_initials(toks):
    out, buf = [], []
    for t in toks:
        if len(t) == 1 and t.isalpha():
            buf.append(t)
            continue
        if buf:
            out.append("".join(buf))
            buf = []
        out.append(t)
    if buf:
        out.append("".join(buf))
    return out

def process_records(df, src):
    recs = []
    for eid, name, addr, country in zip(df["entity_id"], df["business_name"],
                                        df["business_address"], df["country"]):
        nt = merge_initials(tokens(name))
        legal = frozenset(t for t in nt if t in LEGAL)
        core = [t for t in nt if t not in LEGAL and t not in NAME_STOP] or nt
        at = tokens(addr)
        postal = ""
        for t in reversed(at[-4:]):
            if t.isdigit() and len(t) in (5, 6):
                postal = t
                break
        nums = frozenset(t for t in at if any(ch.isdigit() for ch in t) and t != postal)
        recs.append(dict(
            entity_id=eid, src=src, country=country.strip() or "UNK",
            name_norm=" ".join(nt), name_core=" ".join(core), core_set=frozenset(core),
            core_first=core[0] if core else "", legal=legal,
            acronym="".join(t[0] for t in core) if len(core) >= 2 else "",
            name_digits=frozenset(t for t in core if t.isdigit()),
            addr_norm=" ".join(at), addr_set=frozenset(at), postal=postal, nums=nums,
            landmark=int(any(t in LANDMARK for t in at)),
        ))
    return recs

def _chunk_process(args_tuple):
    df_chunk, src = args_tuple
    return process_records(df_chunk, src)

def load_split(data_dir, split, n_jobs=6):
    recs = []
    for k in (1, 2, 3):
        path = os.path.join(data_dir, split, f"{split}_source{k}.tsv")
        df = read_tsv(path)
        
        # Parallelize across 6 CPU cores
        if len(df) > 50000 and n_jobs > 1:
            chunks = np.array_split(df, n_jobs)
            with mp.Pool(n_jobs) as pool:
                res = pool.map(_chunk_process, [(chunk, f"S{k}") for chunk in chunks])
            for r in res:
                recs.extend(r)
        else:
            recs += process_records(df, f"S{k}")
    return pd.DataFrame(recs)

def compute_idf(R):
    idf = {}
    for country, g in R.groupby("country"):
        n = len(g)
        for kind, col in (("name", "core_set"), ("addr", "addr_set")):
            df_ = Counter(t for s in g[col] for t in s)
            idf[(country, kind)] = ({t: math.log((n + 1) / (c + 1)) + 1 for t, c in df_.items()},
                                    math.log(n + 1) + 1)
    return idf

In [3]:
# Cell 3

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor

# Make TT global so worker processes don't need to copy the 5M row matrix
_G_CAND = {}

def _process_chunk(args_tuple):
    i, chunk_size, k = args_tuple
    Q = _G_CAND["Q"]
    TT = _G_CAND["TT"]
    
    # Perform sparse matrix multiplication on just this chunk
    S = (Q[i:i + chunk_size] @ TT).toarray()
    part = np.argpartition(-S, k - 1, axis=1)[:, :k]
    return i, part, np.take_along_axis(S, part, 1)

def topk_cosine(Q, T, k, n_jobs=6):
    k = min(k, T.shape[0])
    nq = Q.shape[0]
    if k == 0 or nq == 0:
        return np.zeros((nq, 0), np.int64), np.zeros((nq, 0), np.float32)
        
    _G_CAND["Q"] = Q
    _G_CAND["TT"] = T.T.tocsc()
    
    # Keep chunks small enough to fit easily in L3 Cache
    chunk = max(10, int(2e7 // max(T.shape[0], 1)))
    idx = np.empty((nq, k), np.int64)
    sim = np.empty((nq, k), np.float32)
    
    # Use ProcessPool to completely bypass Python's GIL and memory leak issues
    tasks = [(i, chunk, k) for i in range(0, nq, chunk)]
    
    with ProcessPoolExecutor(max_workers=n_jobs) as executor:
        # map() iterates safely without exploding memory
        for i, part, s_vals in executor.map(_process_chunk, tasks):
            idx[i:i + chunk] = part
            sim[i:i + chunk] = s_vals
            
    # Clean up global references
    _G_CAND.clear()
    return idx, sim

def generate_candidates(R, args):
    cand = {}
    def add(i, j, flag):
        cand.setdefault((i, j), set()).add(flag)

    cos_store = {}
    n_workers = getattr(args, 'n_jobs', 6)
    
    for country, g in R.groupby("country"):
        s1 = g[g.src == "S1"]
        for src in ("S2", "S3"):
            tg = g[g.src == src]
            if len(s1) == 0 or len(tg) == 0:
                continue
            views = {}
            for view, text_fn in (("name", lambda d: d["name_core"]),
                                  ("full", lambda d: d["name_core"] + " | " + d["addr_norm"])):
                vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3), sublinear_tf=True, dtype=np.float32)
                vec.fit(pd.concat([text_fn(s1), text_fn(tg)]))
                A, B = vec.transform(text_fn(s1)), vec.transform(text_fn(tg))
                views[view] = (A, B)
                kf = args.k_name if view == "name" else args.k_full
                
                idx, _ = topk_cosine(A, B, kf, n_workers)
                for a in range(idx.shape[0]):
                    for b in idx[a]:
                        add(s1.index[a], tg.index[b], "f_" + view)
                
                if args.k_rev > 0:
                    idx, sim = topk_cosine(B, A, args.k_rev, n_workers)
                    for b in range(idx.shape[0]):
                        for a, sv in zip(idx[b], sim[b]):
                            if sv > 0.2:
                                add(s1.index[a], tg.index[b], "f_rev")
                            
            buckets = defaultdict(list)
            for i, p, f in zip(s1.index, s1.postal, s1.core_first):
                if p and f:
                    buckets[(p, f)].append(i)
            for j, p, f in zip(tg.index, tg.postal, tg.core_first):
                b = buckets.get((p, f), [])
                if 0 < len(b) <= 50:
                    for i in b:
                        add(i, j, "f_zip")
            cos_store[(country, src)] = (views, {v: k for k, v in enumerate(s1.index)},
                                         {v: k for k, v in enumerate(tg.index)})

    rows = [(i, j, int("f_name" in f), int("f_full" in f), int("f_zip" in f), int("f_rev" in f))
            for (i, j), f in cand.items()]
    C = pd.DataFrame(rows, columns=["i", "j", "f_name", "f_full", "f_zip", "f_rev"])
    C["cos_name"] = np.nan
    C["cos_full"] = np.nan
    key = list(zip(R.loc[C.i, "country"].values, R.loc[C.j, "src"].values))
    C["_g"] = key
    for g_key, sub in C.groupby("_g"):
        views, m1, m2 = cos_store[g_key]
        ia = np.array([m1[i] for i in sub.i])
        jb = np.array([m2[j] for j in sub.j])
        for view in ("name", "full"):
            A, B = views[view]
            C.loc[sub.index, "cos_" + view] = np.asarray(A[ia].multiply(B[jb]).sum(1)).ravel()
    C = C.drop(columns="_g")
    C["_s"] = C[["cos_name", "cos_full"]].max(axis=1) + 0.05 * C.f_zip
    C = C.sort_values(["i", "_s"], ascending=[True, False])
    C = C.groupby("i", sort=False).head(args.max_cands).drop(columns="_s").reset_index(drop=True)
    return C

In [4]:
#Cell 4

_G = {}

def _idf_overlap(a, b, idf, default):
    if not a and not b:
        return np.nan, np.nan
    w = lambda t: idf.get(t, default)
    inter = sum(w(t) for t in a & b)
    union = sum(w(t) for t in a | b)
    unmatched = max((w(t) for t in a ^ b), default=0.0)
    return inter / union if union else np.nan, unmatched

def _pair_feats(ij):
    R, idf = _G["R"], _G["idf"]
    i, j = ij
    a, b = R[i], R[j]
    n1, n2 = a["name_core"], b["name_core"]
    ad1, ad2 = a["addr_norm"], b["addr_norm"]
    ni, nd = idf[(a["country"], "name")]
    ai, adf = idf[(a["country"], "addr")]
    n_idf, n_unm = _idf_overlap(a["core_set"], b["core_set"], ni, nd)
    a_idf, a_unm = _idf_overlap(a["addr_set"], b["addr_set"], ai, adf)
    cs1, cs2 = a["core_set"], b["core_set"]
    as1, as2 = a["addr_set"], b["addr_set"]
    acr = int(bool(a["acronym"]) and a["acronym"] in cs2) or int(bool(b["acronym"]) and b["acronym"] in cs1)
    if a["postal"] and b["postal"]:
        zip_eq = float(a["postal"] == b["postal"])
        zip_pref = float(a["postal"][:3] == b["postal"][:3])
    else:
        zip_eq = zip_pref = np.nan
    if a["nums"] and b["nums"]:
        num_j = len(a["nums"] & b["nums"]) / len(a["nums"] | b["nums"])
    else:
        num_j = np.nan
    return (
        fuzz.ratio(n1, n2), fuzz.partial_ratio(n1, n2), fuzz.token_set_ratio(n1, n2),
        fuzz.token_sort_ratio(n1, n2), JaroWinkler.normalized_similarity(n1, n2),
        Levenshtein.normalized_similarity(n1, n2),
        fuzz.token_set_ratio(a["name_norm"], b["name_norm"]),
        len(cs1 & cs2) / max(len(cs1 | cs2), 1), n_idf, n_unm,
        int(a["core_first"] == b["core_first"]), acr, len(cs1), len(cs2),
        int(bool(a["legal"]) and bool(b["legal"]) and not (a["legal"] & b["legal"])),
        int(bool(a["name_digits"] ^ b["name_digits"])),
        fuzz.ratio(ad1, ad2), fuzz.partial_ratio(ad1, ad2), fuzz.token_set_ratio(ad1, ad2),
        fuzz.token_sort_ratio(ad1, ad2), len(as1 & as2) / max(len(as1 | as2), 1), a_idf, a_unm,
        zip_eq, zip_pref, num_j, len(as1), len(as2), a["landmark"], b["landmark"],
        int(b["src"] == "S3"),
        fuzz.token_set_ratio(n2, ad1),
    )

PAIR_COLS = ["n_ratio", "n_pratio", "n_tset", "n_tsort", "n_jw", "n_lev", "n_full_tset", "n_jacc",
             "n_idf", "n_idf_unmatched_max", "n_first_eq", "n_acronym", "n_len1", "n_len2",
             "legal_conflict", "name_digit_conflict", "a_ratio", "a_pratio", "a_tset", "a_tsort",
             "a_jacc", "a_idf", "a_idf_unmatched_max", "zip_eq", "zip_prefix_eq", "num_jacc",
             "a_len1", "a_len2", "landmark1", "landmark2", "is_s3", "name_in_addr"]

from concurrent.futures import ProcessPoolExecutor

def _chunk_pair_feats(chunk):
    return [_pair_feats(p) for p in chunk]

def featurize(C, R, idf, n_jobs):
    _G["R"] = R.to_dict("records")
    _G["idf"] = idf
    pairs = list(zip(C.i.values, C.j.values))
    
    # Split across available CPU cores
    n_workers = max(1, n_jobs)
    chunk_size = max(1000, len(pairs) // (n_workers * 4))
    chunks = [pairs[i:i + chunk_size] for i in range(0, len(pairs), chunk_size)]
    
    log(f"  featurizing {len(pairs):,} pairs across {n_workers} CPU cores...")
    out = []
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        for res in executor.map(_chunk_pair_feats, chunks):
            out.extend(res)
            
    F = pd.DataFrame(out, columns=PAIR_COLS, index=C.index).astype(np.float32)
    F = pd.concat([C, F], axis=1)

    F["combo"] = 0.5 * F.cos_full + 0.25 * F.n_tset / 100 + 0.25 * F.a_tset / 100
    for col in ("cos_name", "cos_full", "combo"):
        g1 = F.groupby("i")[col]
        F[f"{col}_rank_s1"] = g1.rank(ascending=False, method="min")
        F[f"{col}_gap_s1"] = g1.transform("max") - F[col]
        g2 = F.groupby("j")[col]
        F[f"{col}_rank_cand"] = g2.rank(ascending=False, method="min")
        F[f"{col}_gap_cand"] = g2.transform("max") - F[col]
    top2 = F.sort_values(["i", "combo"], ascending=[True, False]).groupby("i").head(2).groupby("i")["combo"]
    gap2 = (top2.max() - top2.min()).where(top2.size() > 1, 1.0)
    F["combo_second_gap_s1"] = F.i.map(gap2).values
    F["n_cands_s1"] = F.groupby("i")["j"].transform("size")
    F["n_s1_per_cand"] = F.groupby("j")["i"].transform("size")
    F["mutual_best"] = ((F.combo_rank_s1 == 1) & (F.combo_rank_cand == 1)).astype(np.int8)
    key = R.country + "\x1f" + R.name_core
    freq = key[R.src == "S1"].value_counts()
    F["s1_name_freq"] = key.loc[F.i].map(freq).fillna(1).values
    return F

In [5]:
##Cell 5

LGB_PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=63, min_child_samples=20,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  verbose=-1, seed=SEED)

def train_oof(F, y, groups, n_folds, X_test=None):
    oof = np.zeros(len(F))
    test_pred = np.zeros(len(X_test)) if X_test is not None else None
    gkf = GroupKFold(n_splits=n_folds)
    imp = np.zeros(len(FEATURES))
    for k, (tr, va) in enumerate(gkf.split(F, y, groups)):
        dtr = lgb.Dataset(F.iloc[tr][FEATURES], y[tr])
        dva = lgb.Dataset(F.iloc[va][FEATURES], y[va])
        m = lgb.train(LGB_PARAMS, dtr, 3000, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va] = m.predict(F.iloc[va][FEATURES], num_iteration=m.best_iteration)
        imp += m.feature_importance("gain")
        if X_test is not None:
            test_pred += m.predict(X_test[FEATURES], num_iteration=m.best_iteration) / n_folds
        log(f"  fold {k}: best_iter={m.best_iteration}")
    return oof, test_pred, pd.Series(imp, FEATURES).sort_values(ascending=False)

def f05(pred, gold):
    if not gold and not pred:
        return 1.0
    if not gold or not pred:
        return 0.0
    tp = len(pred & gold)
    return 0.0 if tp == 0 else 1.25 * tp / (0.25 * len(gold) + len(pred))

def macro_f05(pred, gold, s1_ids):
    return float(np.mean([f05(pred.get(s, set()), gold.get(s, set())) for s in s1_ids]))

def apply_exclusivity(D):
    D = D.copy()
    best = D.groupby("cand")["p"].transform("max")
    first = ~D.sort_values("p", ascending=False).duplicated("cand")
    D.loc[~((D.p == best) & first.reindex(D.index)), "p"] = 0.0
    return D

def decide_threshold(D, t1, t2):
    out = {}
    for s, g in D[D.p >= min(t1, t2)].groupby("s1"):
        g = g.sort_values("p", ascending=False)
        if g.p.iloc[0] < t1:
            continue
        out[s] = {g.cand.iloc[0]} | set(g.cand.iloc[1:][g.p.iloc[1:] >= t2])
    return out

def decide_expected_f(D, n_mc=1000, p_floor=0.02, max_k=15, p_unseen=0.0, seed=SEED):
    rng = np.random.default_rng(seed)
    E = D[D.p >= p_floor].sort_values(["s1", "p"], ascending=[True, False])
    E = E[E.groupby("s1").cumcount() < max_k]
    s1v, cv, pv = E.s1.values, E.cand.values, E.p.values
    bounds = np.flatnonzero(np.r_[True, s1v[1:] != s1v[:-1], True])
    out = {}
    for a, b in zip(bounds[:-1], bounds[1:]):
        p = pv[a:b]
        draws = rng.random((n_mc, len(p))) < p
        total = draws.sum(1) + (rng.random(n_mc) < p_unseen)
        tp = np.cumsum(draws, axis=1)
        k = np.arange(1, len(p) + 1)
        denom = 1.25 * tp + 0.25 * (total[:, None] - tp) + (k - tp)
        v = np.where(tp > 0, 1.25 * tp / np.maximum(denom, 1e-9), 0.0).mean(0)
        if v.max() > float(np.mean(total == 0)):
            out[s1v[a]] = set(cv[a:a + int(v.argmax()) + 1])
    return out

def _fast_threshold_scorer(D, gold, s1_ids):
    D = D.sort_values(["s1", "p"], ascending=[True, False]).reset_index(drop=True)
    pos = {s: k for k, s in enumerate(s1_ids)}
    code = D.s1.map(pos).values
    first = (D.groupby("s1").cumcount() == 0).values
    top_p = D.groupby("s1")["p"].transform("first").values
    y = np.array([c in gold.get(s, ()) for s, c in zip(D.s1, D.cand)], dtype=float)
    p = D.p.values
    G = np.array([len(gold.get(s, ())) for s in s1_ids], dtype=float)
    n = len(s1_ids)

    def score(t1, t2):
        sel = (top_p >= t1) & (first | (p >= t2))
        npred = np.bincount(code[sel], minlength=n).astype(float)
        tp = np.bincount(code[sel], weights=y[sel], minlength=n)
        f = np.where(tp > 0, 1.25 * tp / np.maximum(0.25 * G + npred, 1e-9), 0.0)
        f = np.where((G == 0) & (npred == 0), 1.0, f)
        return float(f.mean())
    return score

def search_decisions(D, gold, s1_ids):
    res, variants = [], {}
    for excl in (False, True):
        Dx = apply_exclusivity(D) if excl else D
        tag = "excl" if excl else "raw"
        scorer = _fast_threshold_scorer(Dx, gold, s1_ids)
        grid = [(scorer(t1, t2), (round(float(t1), 3), round(float(t2), 3)))
                for t1 in np.arange(0.15, 0.86, 0.025) for t2 in np.arange(0.3, 0.98, 0.025)]
        sc, (t1, t2) = max(grid)
        name = f"threshold[{tag}] t1={t1} t2={t2}"
        res.append((name, sc))
        variants[name] = (excl, "thr", (t1, t2))
        name = f"expected_f[{tag}]"
        res.append((name, macro_f05(decide_expected_f(Dx), gold, s1_ids)))
        variants[name] = (excl, "expf", None)
    res.sort(key=lambda r: -r[1])
    return res, variants

def run_decision(D, spec):
    excl, kind, params = spec
    Dx = apply_exclusivity(D) if excl else D
    return decide_threshold(Dx, *params) if kind == "thr" else decide_expected_f(Dx)

In [ ]:
# Cell 6


def diagnostics(R, F, gold):
  s1_ids = R.loc[R.src == "S1", "entity_id"].tolist()
  country_of = dict(zip(R.entity_id, R.country))
  gold_pairs = {(s, c) for s, cs in gold.items() for c in cs}
  cand_pairs = set(zip(F.s1, F.cand))
  n_single = sum(1 for s in s1_ids if not gold.get(s))
  log("---- dataset / blocking diagnostics ----")
  log(
      f"  S1 entities: {len(s1_ids):,} | singletons: {n_single:,}"
      f" ({n_single / len(s1_ids):.1%})"
  )
  sizes = Counter(len(gold.get(s, ())) for s in s1_ids)
  log(f"  match-set size distribution: {dict(sorted(sizes.items())[:10])}")
  cross = sum(
      1 for s, c in gold_pairs if country_of.get(s) != country_of.get(c)
  )
  log(
      f"  cross-country gold pairs: {cross} (if 0, per-country blocking is safe)"
  )
  multi = sum(1 for v in Counter(c for _, c in gold_pairs).values() if v > 1)
  log(
      "  S2/S3 records matched to >1 S1:"
      f" {multi} (if 0, exclusivity constraint is valid)"
  )
  rec = len(gold_pairs & cand_pairs) / max(len(gold_pairs), 1)
  ent_full = np.mean([
      gold[s] <= {c for c in gold[s] if (s, c) in cand_pairs}
      for s in s1_ids
      if gold.get(s)
  ])
  log(
      f"  blocking pair recall: {rec:.4f} | entities with ALL matches"
      f" recalled: {ent_full:.4f}"
  )
  for flag in ("f_name", "f_full", "f_zip", "f_rev"):
    sub = set(zip(F.s1[F[flag] == 1], F.cand[F[flag] == 1]))
    log(
        f"    {flag:7s}: recall alone"
        f" {len(gold_pairs & sub) / max(len(gold_pairs), 1):.4f},"
        f" {len(sub):,} pairs"
    )


def score_report(name, pred, gold, s1_ids, country_of):
  msg = f"  {name}: macro F0.5 = {macro_f05(pred, gold, s1_ids):.4f}"
  for c in sorted({country_of[s] for s in s1_ids}):
    ids = [s for s in s1_ids if country_of[s] == c]
    msg += f" | {c}: {macro_f05(pred, gold, ids):.4f}"
  sing = [s for s in s1_ids if not gold.get(s)]
  if sing:
    msg += f" | singletons: {macro_f05(pred, gold, sing):.4f}"
  log(msg)


class PipelineArgs:
  data = "dataset"
  out = "output"
  work = "work_dir"
  mode = "full"
  loco = False
  folds = 5
  k_name = 5  # candidate blocker limits
  k_full = 5
  k_rev = 0
  max_cands = 15  # top-20 pairs per S1
  n_jobs = 20  # all 6 CPU cores
  max_train_s1 = 60000  # train on 60k S1 sample for rapid GBDT convergence


args = PipelineArgs()

# --- 1. TRAIN PIPELINE ---
log(f"loading train data (parallel on {args.n_jobs} cores)...")
R = load_split(args.data, "train", n_jobs=args.n_jobs)
log(
    f"  train: "
    + ", ".join(f"{k}={v}" for k, v in R.groupby(["src"]).size().items())
    + " | countries: "
    + ", ".join(f"{k}={v}" for k, v in R.country.value_counts().items())
)

# Subsample S1 for lightning-fast training without losing GBDT tree quality
if args.max_train_s1 and (R.src == "S1").sum() > args.max_train_s1:
  log(
      f"subsampling train S1 to {args.max_train_s1:,} entities to prevent"
      " memory explosion..."
  )
  s1_sample = R[R.src == "S1"].sample(n=args.max_train_s1, random_state=SEED)
  R = pd.concat([s1_sample, R[R.src != "S1"]]).reset_index(drop=True)

idf = compute_idf(R)
C = generate_candidates(R, args)
log(
    f"  train candidates: {len(C):,} pairs ("
    f"{len(C) / max((R.src == 'S1').sum(), 1):.1f} per S1)"
)

F = featurize(C, R, idf, args.n_jobs)
F["s1"] = R.loc[F.i, "entity_id"].values
F["cand"] = R.loc[F.j, "entity_id"].values
F["country"] = R.loc[F.i, "country"].values
log("  train features calculated")

gt = read_tsv(os.path.join(args.data, "train", "train_ground_truth.tsv"))
gold = {
    s: set(filter(None, m.split(",")))
    for s, m in zip(gt.source1_entity_id, gt.matched_entity_ids)
}
s1_ids = R.loc[R.src == "S1", "entity_id"].tolist()
country_of = dict(zip(R.entity_id, R.country))
diagnostics(R, F, gold)

gold_pairs = {(s, c) for s, cs in gold.items() for c in cs}
y = np.array([(s, c) in gold_pairs for s, c in zip(F.s1, F.cand)], dtype=int)

ignore = {"i", "j", "s1", "cand", "country"}
FEATURES = [c for c in F.columns if c not in ignore]

# --- 2. TEST PIPELINE ---
Ft = None
Rt = None
if args.mode == "full":
  log(f"loading test data (parallel on {args.n_jobs} cores)...")
  Rt = load_split(args.data, "test", n_jobs=args.n_jobs)
  idft = compute_idf(Rt)
  Ct = generate_candidates(Rt, args)
  log(
      f"  test candidates: {len(Ct):,} pairs ("
      f"{len(Ct) / max((Rt.src == 'S1').sum(), 1):.1f} per S1)"
  )
  Ft = featurize(Ct, Rt, idft, args.n_jobs)
  Ft["s1"] = Rt.loc[Ft.i, "entity_id"].values
  Ft["cand"] = Rt.loc[Ft.j, "entity_id"].values
  Ft["country"] = Rt.loc[Ft.i, "country"].values
  log("  test features calculated")

# --- 3. MODEL TRAINING & PREDICTION ---
log("training LightGBM (GroupKFold by S1 entity)...")
oof, test_pred, imp = train_oof(F, y, F.s1.values, args.folds, Ft)
log("  top features: " + ", ".join(imp.index[:12]))

D = pd.DataFrame({"s1": F.s1, "cand": F.cand, "p": oof})
D.to_csv(os.path.join(args.work, "oof_train.tsv"), sep="\t", index=False)

log("---- decision layer (OOF) ----")
res, variants = search_decisions(D, gold, s1_ids)
for name, sc in res:
  log(f"  {sc:.4f}  {name}")
best_name = res[0][0]
score_report(
    f"BEST {best_name}",
    run_decision(D, variants[best_name]),
    gold,
    s1_ids,
    country_of,
)

if args.mode == "full":
  log(f"---- predicting test with: {best_name} ----")
  Dt = pd.DataFrame({"s1": Ft.s1, "cand": Ft.cand, "p": test_pred})
  Dt.to_csv(os.path.join(args.work, "test_scores.tsv"), sep="\t", index=False)
  pred = run_decision(Dt, variants[best_name])
  t_s1 = Rt.loc[Rt.src == "S1", "entity_id"].tolist()
  cands = (
      Dt.sort_values("p", ascending=False)
      .groupby("s1")["cand"]
      .apply(list)
      .to_dict()
  )
  pred = {s: [c for c in cands.get(s, []) if c in m] for s, m in pred.items()}

  write_id_lists(
      os.path.join(args.out, "candidate_pairs.tsv"),
      t_s1,
      cands,
      "candidate_entity_ids",
  )
  write_id_lists(
      os.path.join(args.out, "matching_results.tsv"),
      t_s1,
      pred,
      "matched_entity_ids",
  )

  valid = set(Rt.loc[Rt.src != "S1", "entity_id"])
  assert all(c in valid for v in pred.values() for c in v)
  assert all(set(pred.get(s, [])) <= set(cands.get(s, [])) for s in t_s1)

  n_match = sum(1 for s in t_s1 if pred.get(s))
  t_country = dict(zip(Rt.entity_id, Rt.country))
  log(
      f"  wrote outputs: {n_match:,}/{len(t_s1):,} S1 entities with >=1 match;"
      " by country: "
      + ", ".join(
          f"{c}={np.mean([bool(pred.get(s)) for s in t_s1 if t_country[s] == c]):.2f}"
          for c in sorted(Rt.country.unique())
      )
  )

log("done!")

[    0.7s] loading train data (parallel on 20 cores)...


/home/btech10052.23/new_project/env/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
/home/btech10052.23/new_project/env/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
/home/btech10052.23/new_project/env/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


[  350.4s]   train: S1=2206821, S2=5034616, S3=5285603 | countries: US=7510506, India=5016534
[  351.2s] subsampling train S1 to 60,000 entities to prevent memory explosion...
